# Consolidated sweep - ingest many tables in ONE job task

One `lakefed_ingest_mt_sweep` run processes a whole **task_collection** of (tenant db, table)
tasks in this single notebook, using a thread pool + Spark **FAIR scheduler pools** so each
table's work overlaps on the cluster. This decouples throughput from the workspace job limits
(2,000 concurrent task runs; 10,000 run-submits/hour) that a task-per-table design hits - the
throttle becomes compute (cluster cores), which is the intended cost model. Shard by
`task_collection` for horizontal scale: run one sweep per collection.

Notes:
- **One concurrency knob, `parallelism`** = the most units of query work running at once in this
  run, of ANY type: a CT read+MERGE, a non-partitioned seed, ONE partition of a partitioned seed, a
  small source query (CT version / min-valid / size / bounds), or the staging->target swap. Each unit
  holds one slot of a shared `parallel.WorkSlots(parallelism)`; threads may outnumber slots, but a
  thread only runs Spark/JDBC work while holding a slot, and never holds one while waiting (e.g. a
  table waiting for its partitions) - so nested fan-out can't deadlock. Each slot is its own FAIR pool
  (`pool0..pool{parallelism-1}`). The budget is per run: concurrent sweeps of different collections
  on separate job clusters each have their own.
- Cluster must run the **FAIR** scheduler (`spark.scheduler.mode=FAIR`) for the pools to apply.
- All Change-Tracking / source-touching work is in **ct_engine.py** (Spark JDBC; no SQL warehouse,
  no UC connection) - this notebook is pure orchestration + Lakebase checkpointing + telemetry.
- One source read per table (per range for partitioned seeds), no `.cache()`: the seed streams the
  JDBC read into the Delta write and the CT increment feeds the CHANGETABLE read straight into the
  MERGE. Partitioned seeds load a staging table and swap it into the target in one atomic commit, so
  readers never see a partial table.
- Because the per-table work runs in-process (no Jobs-UI task per table), progress/telemetry is
  written to Lakebase: one `sweep_run` row per run and one `ingest_event` row per (sweep, table),
  queryable live and read directly from the Lakebase UC catalog (see docs/monitoring.md).
- Per-table failures are isolated (one bad table does not fail the collection) and recorded to
  both `ingest_event` (with the stage it failed at) and the `ct_checkpoint` status for triage.

In [ ]:
# Cluster task. Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import parallel

In [ ]:
# Params + connect to Lakebase (SDK-minted OAuth token; no stored password), then open a sweep.
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


task_collection = widget("task_collection", "example_cohort")
db_instance = widget("db_instance", "lakefed-ingest-mt")
db_name = widget("db_name", "lakefed_ingest_mt")
# Max concurrent units of query work in this run (all types share it; see the header).
parallelism = int(widget("parallelism", "32"))
# Source credentials: per-DB JSON secrets in this scope (Databricks- or Azure Key Vault-backed);
# each control row names its own secret_key. TLS trust is configurable (prod: "false").
secret_scope = widget("secret_scope", "lakefed_ingest_mt")
trust_server_certificate = widget("trust_server_certificate", "true")
os.environ["MT_SQL_SECRET_SCOPE"] = secret_scope
os.environ["MT_SQL_TRUST_SERVER_CERT"] = trust_server_certificate
# Single source read per table: Delta MERGE materializes a non-Delta (JDBC) source itself when
# this is "auto" (the default) - we add no .cache() of our own. Logged once for visibility.
# (spark.conf.get validates a supplied default against the allowed values, so read it bare.)
try:
    _materialize = spark.conf.get("spark.databricks.delta.merge.materializeSource")
except Exception:
    _materialize = "auto (unset; runtime default)"
print("spark.databricks.delta.merge.materializeSource =", _materialize)

store = StateStore.connect_lakebase(db_instance, db_name)


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
# Per-collection run lock: different task_collections sweep concurrently; a second run of the SAME
# collection (e.g. a schedule firing while the previous sweep is still going) exits as 'skipped'.
# Session-level advisory lock on a DEDICATED connection held for the whole sweep — released
# automatically if the connection drops, so a crashed run never leaves a stale lock.
lock_store = StateStore.connect_lakebase(
    db_instance, db_name, application_name=ss.lock_application_name(task_collection, job_run_id))
if not lock_store.try_lock_collection(task_collection):
    lock_store.close()
    skipped_id = store.start_sweep(task_collection, job_run_id, cluster_id, parallelism)
    store.finish_sweep(skipped_id, total=0, ok=0, failed=0, status="skipped",
                       detail="another sweep of this task_collection is already running")
    print(f"sweep {skipped_id}: SKIPPED - task_collection {task_collection} is already being swept")
    dbutils.notebook.exit(json.dumps({"sweep_id": skipped_id, "task_collection": task_collection,
                                      "status": "skipped"}))

configs = store.list_task_configs(task_collection)

# Lost-lock guard: if the lock connection drops mid-sweep (network blip, Lakebase restart), the lock
# is released and another run could start. Workers re-check it at most every 30s and stop STARTING
# new tables once it is gone; skipped tables are simply picked up by the next sweep.
import threading, time
_lock_state = {"checked": time.monotonic(), "held": True}
_lock_mu = threading.Lock()


def assert_lock_held(every_s=30):
    with _lock_mu:
        if _lock_state["held"] and time.monotonic() - _lock_state["checked"] >= every_s:
            _lock_state["checked"] = time.monotonic()
            _lock_state["held"] = lock_store.is_alive()
        if not _lock_state["held"]:
            raise RuntimeError("collection lock lost; table deferred to the next sweep")
sweep_id = store.start_sweep(task_collection, job_run_id, cluster_id, parallelism)
print(f"sweep {sweep_id}: collection {task_collection} -> {len(configs)} tables (parallelism={parallelism})")

In [ ]:
# Per-(tenant db, table) state machine with Lakebase telemetry. CT/source ops live in ct_engine
# (Spark JDBC); everything here is orchestration + checkpointing + telemetry. All checkpoint /
# status / reseed state is keyed by the control task id (cfg["id"]).
# The `stage` variable records how far a table got, so a failure pinpoints read vs merge vs
# checkpoint. The collection lock is re-checked (cached, <=1 query/30s) right before every commit
# point — seed, each partition + the swap (via `guard`), MERGE, reseed enqueue, checkpoint advance —
# so a run that lost its lock stops writing; skipped work is redone by the next sweep (at-least-once).
# `action` is provisional (seed vs increment) from phase; a cdc->reseed is finished
# as status 'skipped' / action 'reseed' and counted in the roll-up (reseed_enqueued) + reseed_queue.
from functools import partial


def ingest_one(cfg, sweep_id):
    assert_lock_held()
    db, schema, table = cfg["src_database"], cfg["src_schema"], cfg["src_table"]
    sink = f"{cfg['sink_catalog']}.{cfg['sink_schema']}.{cfg['sink_table']}"
    pool = ""  # FAIR pools follow work slots (one per unit); the ones used are recorded at finish
    slots.start_tracking()

    cp = store.get_checkpoint(cfg["id"])
    phase = cp["phase"]
    v_from = cp["ct_version"]
    action = "seed" if ss.decide_action(phase, v_from, None) == "seed" else "increment"
    event_id = store.record_event_start(sweep_id, cfg, action, phase, pool)

    stage = "ensure_sink"
    rows_read = rows_merged = v_to = None
    try:
        ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)

        if action == "seed":
            stage = "read"
            v0 = ct_engine.current_version(cfg, slots=slots)  # capture BEFORE the seed
            stage = "seed"
            assert_lock_held()
            seeded = ct_engine.seed(spark, cfg, sink, slots=slots, executor=unit_executor,
                                    guard=assert_lock_held)
            rows_merged = seeded if isinstance(seeded, int) else None
            stage = "checkpoint"
            assert_lock_held()
            store.advance_version(cfg, ct_version=v0, phase=ss.CDC, detail="seed complete")
            v_to = v0
            outcome = "seeded"
        else:
            stage = "read"
            # Schema drift: select_list gained a column the sink lacks (e.g. the source table added
            # one). CT only returns CHANGED rows, so an increment would leave historical rows NULL in
            # the new column -> reseed instead; the seed's schema-evolving overwrite adds the column.
            new_cols = ct_engine.missing_sink_columns(cfg["select_list"], spark.table(sink).columns)
            mv = None if new_cols else ct_engine.min_valid_version(cfg, slots=slots)
            if new_cols or ss.decide_action(phase, v_from, mv) == "reseed":
                stage = "checkpoint"
                assert_lock_held()
                reason = "schema_change" if new_cols else "retention_miss"
                store.enqueue_reseed(cfg, reason=reason)  # flips phase -> seeding
                if new_cols:
                    print(f"[schema] {db}.{schema}.{table}: sink lacks {new_cols} -> reseed")
                outcome = "reseed_enqueued"
                action = "reseed"
            else:
                new_version = ct_engine.current_version(cfg, slots=slots)  # upper bound BEFORE the read
                stage = "merge"
                assert_lock_held()
                merged = ct_engine.read_and_merge_ct(spark, cfg, sink, v_from, slots=slots)
                rows_merged = merged if isinstance(merged, int) else None
                stage = "checkpoint"
                assert_lock_held()
                store.advance_version(cfg, ct_version=new_version)  # only AFTER commit
                v_to = new_version
                outcome = "incremented"

    except Exception as e:
        err = str(e)
        pool = _pools_label(cfg, action)
        try:
            store.record_event_finish(event_id, "failed", stage=stage, rows_read=rows_read,
                                      rows_merged=rows_merged, ct_version_from=v_from,
                                      ct_version_to=v_to, error=err, pool=pool)
        except Exception:
            pass
        try:
            store.set_status(cfg, "failed", err[:500])
        except Exception:
            pass
        print(f"[failed@{stage}] {db}.{schema}.{table} pool={pool}: {err[:200]}")
        raise

    # Success-path telemetry runs AFTER the try: the ingest (and its checkpoint) already committed,
    # so a telemetry write error must not flip this table to failed — log it and move on.
    # A reseed copied no data: record it as 'skipped' (action 'reseed') so last_success_at is NOT
    # stamped and the freshness/lag view keeps showing the table as stale until it re-seeds.
    pool = _pools_label(cfg, action)
    try:
        store.record_event_finish(event_id, "skipped" if action == "reseed" else "ok", stage=stage,
                                  rows_read=rows_read, rows_merged=rows_merged,
                                  ct_version_from=v_from, ct_version_to=v_to, action=action, pool=pool)
    except Exception as te:
        print(f"[telemetry] {db}.{schema}.{table}: record_event_finish failed (ingest unaffected): {str(te)[:200]}")
    print(f"[ok:{outcome}] {db}.{schema}.{table} pool={pool}")
    return outcome


def _pools_label(cfg, action):
    """FAIR pools this table's own units ran in (e.g. 'pool3,pool17'). A partitioned seed's partitions
    run on the shared executor in whatever pools are free, so they're summarized, not listed."""
    used = ",".join(slots.tracked_pools()) or "-"
    return used + ("+partitions" if action == "seed" and cfg.get("load_partitioned") else "")


worker = partial(ingest_one, sweep_id=sweep_id)


def set_pool(pool):
    # Called when a unit acquires a slot: that unit's Spark jobs run in the slot's FAIR pool.
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", pool)


# ONE budget for all query work in this run, plus a shared executor that runs partition units.
# Table workers (run_parallel below) only orchestrate: they hold a slot just while doing a unit
# themselves and never while waiting on their partitions, so slots can't deadlock.
from concurrent.futures import ThreadPoolExecutor
slots = parallel.WorkSlots(parallelism, set_pool=set_pool)
unit_executor = ThreadPoolExecutor(max_workers=parallelism, thread_name_prefix="mt-unit")

In [ ]:
# Sweep the collection: table workers orchestrate; every unit of query work draws on `slots`.
results = parallel.run_parallel(
    configs, worker, parallelism, set_pool=None, item_id=lambda c: c["id"]
)
unit_executor.shutdown(wait=True)
summary = parallel.summarize(results)
by = summary.get("by_outcome", {})
reseeded = by.get("reseed_enqueued", 0)
skipped = by.get("skipped", 0)  # reserved for a future skip-no-change outcome
store.finish_sweep(
    sweep_id,
    total=summary["total"], ok=summary["ok"], failed=summary["failed"],
    skipped=skipped, reseeded=reseeded,
    status="completed" if summary["failed"] == 0 else "failed",
    detail=json.dumps({"parallelism": parallelism, "peak_units": slots.peak}),
)

summary["sweep_id"] = sweep_id
summary["task_collection"] = task_collection
summary["parallelism"] = parallelism
summary["peak_units"] = slots.peak
print(json.dumps(summary, indent=2))

failures = [r for r in results if r["status"] == "failed"]
if failures:
    print("failures:", json.dumps(failures, default=str)[:2000])

lock_store.close()  # closing the session releases the collection lock (safe even if it already dropped)

dbutils.notebook.exit(json.dumps(summary, default=str))